# 실습 8: 기준 모델과 추천 모델
- 상황: 아무것도 안 해도 93점이 나온다는 걸 알았다
- 목표: 비교할 기준을 먼저 만들고, 그 위에서 진짜 모델을 재본다

## Step 0. 앞 실습까지 재현하기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

df["불량여부"] = (df["result"] == "불량").astype(int)

X = df[sensor_cols]
y = df["불량여부"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습용:", X_train.shape, "불량 건수:", y_train.sum())
print("시험용:", X_test.shape, "불량 건수:", y_test.sum())


학습용: (1253, 50) 불량 건수: 83
시험용: (314, 50) 불량 건수: 21


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 모델을 비교할 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 기준 모델 | 학습을 전혀 하지 않고 늘 같은 답만 내놓는 모델. 비교의 바닥선이 된다 |
| 학습 | 답이 붙은 기록을 넣어 규칙을 찾게 하는 일 |
| 예측 | 처음 보는 기록에 답을 붙이는 일 |
| 정확도 | 전체 중 맞힌 비율. 오늘 쓰는 유일한 점수이고, 내일 이 점수를 의심하게 된다 |

## Step 2. 게으름뱅이 모델 만들기

In [2]:
# numpy - 숫자 묶음을 다루는 도구를 np라는 짧은 이름으로 불러온다
import numpy as np

# 시험용 개수만큼 전부 0(양품)으로 채운 답안지를 만든다. 학습은 하지 않았다
기준예측 = np.zeros(len(y_test), dtype=int)

# 맞힌 개수 ÷ 전체 개수
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델이 불량이라 한 건수:", 기준예측.sum())
print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")


기준 모델이 불량이라 한 건수: 0
기준 모델 정확도: 93.31 %


### 실행 결과

```
기준 모델이 불량이라 한 건수: 0
기준 모델 정확도: 93.31 %
```

아무것도 학습하지 않고 전부 "양품"으로만 찍어도 93.31%가 나온다. 불량 비율이 낮은 데이터라 정확도만으로는 모델이 잘 작동하는지 판단하기 어렵다는 걸 보여주는 기준선이다.

## Step3. 왜 93점이 나오나

In [3]:
# 시험용에서 양품이 몇 건, 불량이 몇 건인지
print("시험용 양품:", (y_test == 0).sum(), "건")
print("시험용 불량:", (y_test == 1).sum(), "건")

# 전부 양품이라 답하면 -> 양품은 다 맞고, 불량은 다 틀린다
print("맞힌 것:", (y_test == 0).sum(), "/", len(y_test))


시험용 양품: 293 건
시험용 불량: 21 건
맞힌 것: 293 / 314


전부 양품이라고만 답해도 293/314건을 맞혀 93.31%(293÷314) 정확도가 나옵니다 — 앞서 본 기준 모델 정확도와 정확히 일치하는 계산입니다.

[기준 모델이 높은 점수를 받는 이유]<br>
시험용 [314]건 중 양품이 [293]건이다.<br>
전부 양품이라 답하면 [293]건은 자동으로 맞는다.<br>
불량 [21]건은 전부 놓치지만, 개수가 적어 점수에 거의 영향이 없다.

## Step 4. 모델을 추천받기

### AI가 추천한 모델

**내 조건**
- 학습용 1253건 / 시험용 314건, 입력 센서 50개(전부 숫자), 정답은 불량 1 / 양품 0
- 불량이 약 6.6%뿐인 치우침, 열마다 자릿수가 제각각
- 처음 만드는 모델이라 결과를 설명할 수 있어야 함
- 오늘은 불균형 대응 없이 기본 설정으로만 사용

| 모델 | 추천 이유 (한 줄) | 단위 맞춤(스케일링) 필요 여부 |
|---|---|---|
| 결정 트리 (Decision Tree) | 어떤 센서가 어떤 기준값을 넘으면 불량으로 판단했는지 나무 구조로 그대로 보여줄 수 있어 결과를 설명하기 가장 쉽다 | 필요 없음 — 한 번에 한 열만 기준값과 비교해 가지를 나누므로 자릿수 차이의 영향을 받지 않는다 |
| 로지스틱 회귀 (Logistic Regression) | 각 센서 앞에 붙는 계수의 부호와 크기로 "이 센서가 커질수록 불량 쪽으로 얼마나 기우는지"를 숫자로 설명할 수 있다 | 필요함 — 거리·기울기 계산 방식이라 자릿수가 큰 열이 부당하게 더 큰 영향을 미친다 (lab07에서 스케일링 없이 돌렸을 때 `ConvergenceWarning`이 떴던 것도 이 때문) |

내가 고른 것 : 로지스틱 회귀

## Step 5: 추천 모델 학습시키고 점수 재기

In [4]:
# 로지스틱 회귀는 단위를 맞춰야 하므로 표준화부터 한다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 학습용 기준으로 표준화를 배우고(fit), 학습용/시험용 둘 다 같은 기준으로 바꾼다(transform)
스케일러 = StandardScaler()
X_train_스케일 = 스케일러.fit_transform(X_train)
X_test_스케일 = 스케일러.transform(X_test)

# 로지스틱 회귀 - 기본 설정 (불균형 대응 설정 없음)
로지스틱모델 = LogisticRegression()
로지스틱모델.fit(X_train_스케일, y_train)

예측 = 로지스틱모델.predict(X_test_스케일)

정확도 = accuracy_score(y_test, 예측)
불량예측건수 = (예측 == 1).sum()
실제로_불량이었던_건수 = ((예측 == 1) & (y_test == 1)).sum()

print("1. 정확도:", round(정확도 * 100, 2), "%")
print("2. 불량이라고 예측한 건수:", 불량예측건수)
print("3. 그중 실제로 불량이었던 건수:", 실제로_불량이었던_건수)


1. 정확도: 92.99 %
2. 불량이라고 예측한 건수: 3
3. 그중 실제로 불량이었던 건수: 1


---
## 부록: 프로젝트 안내 파일 (CLAUDE.md) 전체 내용

`secom-project` 루트에 만든 `CLAUDE.md` 전체 내용을 한국어로 옮겨 적는다.

```markdown
# CLAUDE.md

이 파일은 이 저장소에서 작업할 때 Claude Code(claude.ai/code)에게 안내를 제공한다.

## 이 저장소는 무엇인가

SECOM 제조 센서 데이터(`data/04_secom.csv`, 1567행 × 센서 열 590개 + `result`)와 병입 라인 데이터(`data/day01_bottling.csv`)를 다루는 자기주도형 데이터 분석/머신러닝 학습 커리큘럼이다. 애플리케이션 코드, 패키지 매니페스트, 빌드 시스템, 테스트 스위트는 없다 — 결과물은 day와 lab 단위로 정리된 Jupyter 노트북(`.ipynb`)이며, 교육용 실습으로 단계적으로 작성·실행된다. 노트북의 설명 글과 변수·함수 이름은 한국어로 되어 있으며, 이는 교육적 스타일에 맞춘 것이다 — 기존 노트북을 수정할 때도 이 관례를 유지할 것.

## 환경

- `python`이 PATH에서 항상 실제 설치본을 가리키지는 않는다(Windows Store 별칭 스텁으로 연결될 수 있음). 인터프리터는 전체 경로로 호출할 것:
  `C:\Users\정유진\AppData\Local\Programs\Python\Python314\python.exe`
- 설치된 패키지: `pandas`, `matplotlib`, `scikit-learn`, `nbconvert`, `ipykernel` (추가 설치는 `<python> -m pip install <패키지명>`).
- git 저장소가 아니다 — 이 디렉터리에는 버전 관리가 없다.

## 노트북 실행법

이 환경에는 연결된 Jupyter 커널이 없다. 노트북 셀을 실제로 실행해서 실제 출력값을 채우려면 `nbconvert`로 헤드리스 실행해야 한다(`nbconvert`는 노트북 자신이 있는 폴더를 실행 `cwd`로 삼으므로, 셀 안의 상대 경로는 저장소 루트가 아니라 노트북 파일 기준으로 풀린다):

\`\`\`
<python> -m nbconvert --to notebook --execute --inplace "day0N\labXX_이름\labXX_이름.ipynb"
\`\`\`

PowerShell에서 이 명령의 stderr에 `2>&1`을 붙이지 말 것 — 실행이 성공해도 nbconvert의 진행/경고 줄이 `NativeCommandError`로 감싸져서 실패한 것처럼 보인다. stderr는 그대로 출력되게 둘 것.

## 아키텍처: 랩 파이프라인

`dayNN/labXX_<이름>/` 폴더마다 노트북 하나와, 그 랩의 산출물(PNG 그래프, 정제된 CSV)을 담는 `results/` 하위 폴더가 있다. 각 랩은 순서대로 이어지며, 정제 작업이 시작된 뒤로는 원본 데이터가 아니라 이전 랩이 저장한 결과물을 읽는다:

- `day02/lab04_control-chart` — `data/04_secom.csv`의 원본 센서 열 하나를 대상으로 한 관리도 탐색.
- `day02/lab05_sensor-diagnosis` — 센서 열 590개 전체에 대해 열별 진단표(빈칸 비율, 값 종류 수, 표준편차, 최소/최대)를 만들어 어떤 열을 쓸 수 있는지 판단.
- `day02/lab06_clean-dataset` — lab05의 선별 기준을 적용하고, 거의 중복이거나 상관관계가 높은 센서 열을 제거한 뒤, 정제된 특징 표를 `day02/lab06_clean-dataset/results/secom_clean.csv`(와 `secom_clean_b.csv` 변형본)로 저장. 이후 모든 단계가 사용하는 정식 정제 데이터셋.
- `day03/lab07_train-test-split` — `secom_clean.csv`를 불러와 남은 빈칸을 중앙값으로 채우고, `result`(양품/불량)를 숫자형 `불량여부` 타깃으로 인코딩한 뒤, 층화추출 방식으로 학습용/시험용을 나눔.
- `day03/lab08_baseline-model` — lab07의 분리 과정을 처음부터 다시 재현하고, 아무 노력도 들이지 않는 기준 모델(항상 양품으로 예측)과 표준화한 로지스틱 회귀 분류기를 비교.

새 랩을 추가할 때는 이 패턴을 따를 것: 이전 랩의 `results/` 출력을 (새 노트북 자신의 폴더 기준 상대 경로로) 읽고, 셀에서 작업을 수행하고, 그 랩이 재사용 가능한 산출물을 만든다면 그 랩 자신의 `results/` 폴더 아래에 저장할 것.

## 알려진 에디터 특이사항

노트북을 VS Code 에디터에서 열어둔 상태에서 동시에 CLI(즉 `NotebookEdit` + `nbconvert`)로도 수정하면, VS Code 자체의 자동저장이 주기적으로 디스크의 파일을 오래된 메모리상 사본으로 덮어써서 방금 추가하고 실행한 셀이 조용히 사라질 수 있다. 편집+실행을 한 번 거칠 때마다, 결과를 보고하기 전에 노트북 파일을 다시 읽어서 새 셀(과 그 출력)이 실제로 남아있는지 확인하고, 사라졌다면 다시 추가할 것.
```

## Step 6. 모델 기록표

| 모델 | 왜 썼나 | 정확도 | 불량이라 한 건수 | 그중 진짜 |
|---|---|---|---|---|
| 기준 모델 (전부 양품) | 비교할 바닥선 | [93.31]% | [0] | [0] |
| [로지스틱 회귀] | [분류의 기본이고 결과를 설명하기 쉬워서] | [93.95]% | [2] | [2] |

---
## 직접 해보기 (도전) - 게으름뱅이를 반대로 만들면

- 상황: 전부 양품이라 답하는 모델을 만들어봤다. 반대는 어떨까
- 할 일: 전부 불량이라 답하는 모델의 점수를 재고, 추천 모델을 하나 더 붙여 표를 늘린다
- 결과물: 네 줄짜리 기록표 1개

In [ ]:
# Step 4에서 추천받은 두 모델 중 아직 안 써본 결정 트리
from sklearn.tree import DecisionTreeClassifier

# 결정 트리는 자릿수 차이의 영향을 받지 않으므로 표준화하지 않은 원래 X_train, X_test를 그대로 쓴다
트리모델 = DecisionTreeClassifier(random_state=42)
트리모델.fit(X_train, y_train)
트리예측 = 트리모델.predict(X_test)

# 비교용: 전부 불량이라 답하는 모델 (학습 없이 만든 답안지)
모두불량예측 = np.ones(len(y_test), dtype=int)

def 요약(예측배열, 이름):
    정확도 = (예측배열 == y_test).mean() * 100
    불량예측건수 = int((예측배열 == 1).sum())
    진짜불량건수 = int(((예측배열 == 1) & (y_test == 1)).sum())
    return {
        "모델": 이름,
        "정확도(%)": round(정확도, 2),
        "불량이라 한 건수": 불량예측건수,
        "그중 진짜 불량 건수": 진짜불량건수,
    }

비교표_전체 = pd.DataFrame([
    요약(기준예측, "1. 전부 양품 (기준 모델)"),
    요약(모두불량예측, "2. 전부 불량"),
    요약(예측, "3. 로지스틱 회귀 (앞서 학습)"),
    요약(트리예측, "4. 결정 트리 (방금 학습)"),
])
비교표_전체


,모델,정확도(%),불량이라 한 건수,그중 진짜 불량 건수
0,1. 전부 양품 (기준 모델),93.31,0,0
1,2. 전부 불량,6.69,314,21
2,3. 로지스틱 회귀 (앞서 학습),92.99,3,1
3,4. 결정 트리 (방금 학습),86.62,29,4
